In [17]:

import pandas as pd
import numpy as np

# Synthetic patient cohort generator for a 21+ female cohort
# Output fields:
#   Patient ID, Age (days), Race, Ethnicity, Smoking, Alcohol, Blood Type, BMI,
#   Has Cervix, HIV/Immunocompromised, Prior Cancer Patient, Family History
#
# This is a first-pass synthetic generator calibrated to public benchmarks.
# The newer binary health-history fields are generated from public age-based
# and race/ethnicity-adjusted prevalence anchors:
#   - Has Cervix: 1 - hysterectomy prevalence
#   - HIV/Immunocompromised: broad immunosuppression proxy
#   - Prior Cancer Patient: self-reported history of cancer
#   - Family History: any first-degree relative with cancer at any site
#
# Important note:
# "HIV/Immunocompromised" is modeled here from public immunosuppression rates
# because NHIS directly measures weakened immune system status. It is therefore
# best interpreted as a broad binary proxy rather than a pure HIV prevalence flag.

# ----------------------------
# Age distribution: NY female, 21+, from Census backbone
# Source page for the population-estimate files used here:
# https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-detail.html
# Dataset names used from that page:
#   - SC-EST2024-AGESEX-CIV  (civilian population by single year of age and sex)
#   - SC-EST2024-SR11H       (population by sex, race, and Hispanic origin)
# How this section is used:
#   1) keep only New York females age 21+
#   2) convert each age count into a probability by dividing by the 21+ total
#   3) sample age in years from that empirical distribution
#   4) convert age in years to age in days later in generate_patients()
# ----------------------------
AGE_VALUES = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85]
AGE_PROBS = np.array([0.015909975693027187, 0.015514818723877663, 0.016165837946892232, 0.01689024862027892, 0.01699947013835047, 0.017212975882406, 0.017355724578320054, 0.017534542152345977, 0.01772477427008793, 0.01796727563118836, 0.017869788708665356, 0.017924952319437078, 0.017984389970164453, 0.01800159302518125, 0.01741507874311268, 0.01717177811225559, 0.016891975474882467, 0.016597230133555383, 0.016439746732069248, 0.01589917571961944, 0.016007676438554838, 0.015882613683578266, 0.01561910688061082, 0.015642815541554155, 0.015150988805421506, 0.014800490746677025, 0.014795095607926827, 0.01443569931312958, 0.01465963231536682, 0.014318527774542467, 0.014495151642568842, 0.015190006179027459, 0.01629945044323262, 0.016382292266177494, 0.015931279440770407, 0.015814107901489253, 0.016029360010582517, 0.0164331408655305, 0.017043768053761793, 0.017272201799168498, 0.017123462119641668, 0.01694025841256843, 0.0168197626872114, 0.01658451968484241, 0.01618842115724186, 0.015949102506540596, 0.015597560630433698, 0.014934859031132527, 0.014587727036858528, 0.013939431938424342, 0.013283166298213077, 0.012806339839006291, 0.012359378318188544, 0.011916549151198764, 0.01150825888909235, 0.011151929991680755, 0.011483318856435978, 0.008466537112776232, 0.008097424134650845, 0.007750418703502376, 0.007755189543798091, 0.006740173239321621, 0.005973056389805381, 0.005479500935361273, 0.03647309388037247], dtype=float)
AGE_PROBS = (AGE_PROBS / AGE_PROBS.sum()).tolist()

# ----------------------------
# Ethnicity / race parameters
# Source page:
# https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-detail.html
# Dataset name used for these proportions:
#   - SC-EST2024-SR11H
# How these probabilities were built:
#   1) pull New York female resident-population counts age 21+ by Hispanic origin and race
#   2) compute P(Hispanic) and P(Non-Hispanic) from those age-eligible female totals
#   3) within Hispanic and Non-Hispanic groups, compute race shares using the official 6-race-group Census product
#
# Important conceptual note:
#   - Hispanic / Latino is an ethnicity, not a race, in Census standards.
#   - People who identify as Hispanic may be of any race.
#   - To avoid treating 'Latino' as a race, this script stores ethnicity separately from race.
#
# Official source pages:
#   - Census explanation of Hispanic origin: https://www.census.gov/data/academy/resources/one-pagers/hispanic-origin.html
#   - Census FAQ on race and ethnicity: https://www.census.gov/programs-surveys/decennial-census/decade/2020/planning-management/release/faqs-race-ethnicity.html
#   - Census state characteristics page with the SC-EST2024-ALLDATA6 dataset:
#     https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-detail.html
#
# Race categories used here come from the Census 6-race-group product:
#   White, Black, American Indian and Alaska Native (AIAN), Asian,
#   Native Hawaiian and Other Pacific Islander (NHPI), and Two or More Races.
# ----------------------------
P_HISPANIC = 2030943 / 10161956
P_NON_HISPANIC = 8131013 / 10161956

RACE_PROBS_NON_HISP = {
    "White": 5361976 / 8131013,
    "Black": 1504583 / 8131013,
    "AIAN": 30664 / 8131013,
    "Asian": 1023010 / 8131013,
    "NHPI": 5269 / 8131013,
    "Two or More": 205511 / 8131013,
}

RACE_PROBS_HISP = {
    "White": 1460416 / 2030943,
    "Black": 360471 / 2030943,
    "AIAN": 83440 / 2030943,
    "Asian": 23744 / 2030943,
    "NHPI": 9962 / 2030943,
    "Two or More": 92910 / 2030943,
}

# ----------------------------
# Smoking and alcohol parameters (New York State, adults 18+)
#
# Official statewide sources used:
# - Smoking: NYS BRFSS 2023 cigarette smoking brief
# - Alcohol: NYS BRFSS / NYS Prevention Agenda 2023 binge-or-heavy drinking
#
# Important note:
# - P_SMOKING below is a statewide adult smoking prevalence.
# - P_ALCOHOL below is NOT "any drinking".
#   It is excessive alcohol use (binge or heavy drinking).
# ----------------------------

# Overall NYS adults. 18+
P_SMOKING = 0.109
P_ALCOHOL = 0.13            #Figure 3    

# ----------------------------
# Blood type parameters
# Source page:
# '1 in 3' and '1 in 16' into numeric probabilities:
# https://www.redcrossblood.org/local-homepage/news/article/know-your-blood-type-facts.html
# How these are used:
#   - convert the approximate U.S. blood-type frequencies into probabilities
#   - normalize them so they sum to 1
#   - sample a categorical blood type for each patient
# Note: these are national blood-type frequencies, not New York-specific frequencies.
# Note: these are estimates from the general population, as age and sex are generally independent from blood types
# ----------------------------
BLOOD_TYPE_PROBS = {
    "O+": 0.3806389033569954,
    "O-": 0.07011716693365753,
    "A+": 0.33389127009166034,
    "A-": 0.06260461333362278,
    "B+": 0.09015064320041683,
    "B-": 0.017529291733414383,
    "AB+": 0.03756276800017368,
    "AB-": 0.007505343350058754,
}

# ----------------------------
# New binary-feature calibration tables
# ----------------------------

# 1) Has Cervix = 1 - hysterectomy prevalence
# Source paper / dataset discussion:
# https://stacks.cdc.gov/view/cdc/113157/cdc_113157_DS1.pdf
# Underlying surveys discussed in that paper:
#   - BRFSS: https://www.cdc.gov/brfss/index.html
#   - NHIS:  https://www.cdc.gov/nchs/nhis/index.html
# How this table is used:
#   - the script stores age-band-specific hysterectomy prevalence by demographic group
#   - later, P(Has Cervix = 1) is set to 1 - P(Hysterectomy = 1)
#   - then a Bernoulli draw is taken for each patient
# Values stored below are proportions, not percentages.
HYSTERECTOMY_BRFFS_2018 = {
    "Hispanic": {
        "21-29": 0.004, "30-39": 0.029, "40-49": 0.109,               # Page 6
        "50-59": 0.211, "60-69": 0.295, "70+": 0.430,
    },
    "White": {
        "21-29": 0.005, "30-39": 0.054, "40-49": 0.166,
        "50-59": 0.268, "60-69": 0.341, "70+": 0.456,
    },
    "Black": {
        "21-29": 0.003, "30-39": 0.038, "40-49": 0.185,
        "50-59": 0.337, "60-69": 0.441, "70+": 0.521,
    },
    "Asian": {
        "21-29": 0.004, "30-39": 0.006, "40-49": 0.078,
        "50-59": 0.112, "60-69": 0.149, "70+": 0.276,
    },
    "Other": {
        "21-29": 0.004, "30-39": 0.038, "40-49": 0.143,
        "50-59": 0.239, "60-69": 0.310, "70+": 0.418,
    },
}

# 2) HIV/Immunocompromised
# Source article based on the 2021 National Health Interview Survey (NHIS):
# https://pmc.ncbi.nlm.nih.gov/articles/PMC10870224/
# How this table is used:
#   - use age-specific immunosuppression prevalence as the base probability
#   - multiply by a race/ethnicity relative-prevalence factor from the same study
#   - draw a Bernoulli outcome for each patient
# Important note:
#   this field is currently a broad immunocompromised proxy, not a pure HIV-only flag.
# First-pass proxy uses 2021 NHIS self-reported immunosuppression rates by age,
# then applies race/ethnicity multipliers from the same study.
IMMUNO_AGE_BASE = {
    "21-29": 0.033,
    "30-39": 0.045,
    "40-49": 0.066,
    "50-59": 0.087,
    "60-69": 0.095,
    "70-79": 0.089,
    "80+": 0.066,
}

IMMUNO_SEX_MULTIPLIER = {
    "Male": 5.2 / 6.6,
    "Female": 7.9 / 6.6,
}

IMMUNO_ETHNICITY_RACE_MULTIPLIER = {
    "Hispanic": 5.0 / 6.6,
    "NonHispanic_White": 7.4 / 6.6,
    "NonHispanic_Black": 6.1 / 6.6,
    "NonHispanic_Asian": 3.7 / 6.6,
    "NonHispanic_Other": 4.2 / 6.6,
    "NonHispanic_AIAN": 8.4 / 6.6,
}

IMMUNO_GROUP_MULTIPLIER = {
    "Hispanic": IMMUNO_SEX_MULTIPLIER["Female"] * IMMUNO_ETHNICITY_RACE_MULTIPLIER["Hispanic"],
    "White": IMMUNO_SEX_MULTIPLIER["Female"] * IMMUNO_ETHNICITY_RACE_MULTIPLIER["NonHispanic_White"],
    "Black": IMMUNO_SEX_MULTIPLIER["Female"] * IMMUNO_ETHNICITY_RACE_MULTIPLIER["NonHispanic_Black"],
    "Asian": IMMUNO_SEX_MULTIPLIER["Female"] * IMMUNO_ETHNICITY_RACE_MULTIPLIER["NonHispanic_Asian"],
    "Other": IMMUNO_SEX_MULTIPLIER["Female"] * IMMUNO_ETHNICITY_RACE_MULTIPLIER["NonHispanic_Other"],
}

# 3) Prior Cancer Patient
# Official source pages:
#   - CDC topic page: https://www.cdc.gov/nchs/hus/topics/history-of-cancer.htm
#     History of Cancer
#   - Direct table PDF:
#     Health, United States 2020–2021, Table CanHst
#
# How this table is used:
#   - start from age-specific prevalence of self-reported history of cancer
#   - optionally multiply by a demographic relative-prevalence factor
#   - draw a Bernoulli outcome for each patient
#
# Important note:
#   - age pattern below comes from 2019 percentages on the CDC History of Cancer page
#   - group multipliers below are based on 2019 age-adjusted percentages from Table CanHst
#   - this is a rough marginal approximation, not a joint age x race model

PRIOR_CANCER_AGE_BASE = {
    "21-44": 0.025,
    "45-54": 0.067,
    "55-64": 0.114,
    "65-74": 0.165,
    "75+":   0.229,
}

PRIOR_CANCER_GROUP_MULTIPLIER = {
    "Hispanic": 4.6 / 6.6,
    "White":    7.5 / 6.6,
    "Black":    4.9 / 6.6,
    "Asian":    2.4 / 6.6,
    "Other":    1.0,
}




# ----------------------------
# Insurance Status (B27001: insured vs uninsured)
# Female-only, age 21+
# Counts taken from the screenshoted ACS B27001 table
# https://data.census.gov/table/ACSDT1Y2022.B27001
# ----------------------------

B27001_FEMALE_COUNTS = {
    "21-25": {"Insured": 13190608, "Uninsured": 1872271},
    "26-34": {"Insured": 18118047, "Uninsured": 2411358},
    "35-44": {"Insured": 20435569, "Uninsured": 2248259},
    "45-54": {"Insured": 18622090, "Uninsured": 1805871},
    "55-64": {"Insured": 19717410, "Uninsured": 1476065},
    "65-74": {"Insured": 18473038, "Uninsured": 179909},
    "75+":   {"Insured": 13985406, "Uninsured": 75089},
}

def build_insurance_prob_table(counts_dict):
    probs = {}
    for band, vals in counts_dict.items():
        insured = vals["Insured"]
        uninsured = vals["Uninsured"]
        total = insured + uninsured
        probs[band] = {
            "Insured": insured / total,
            "Uninsured": uninsured / total,
        }
    return probs

INSURANCE_PROBS = build_insurance_prob_table(B27001_FEMALE_COUNTS)

# 4) Family History
# Source article based on the 2015 National Health Interview Survey (NHIS):
# https://pmc.ncbi.nlm.nih.gov/articles/PMC9162122/
# How this table is used:
#   - start from age-specific prevalence of reporting at least one first-degree relative with cancer
#   - multiply by a demographic relative-prevalence factor
#   - draw a Bernoulli outcome for each patient
# This is intentionally general family history of cancer, not site-specific family history.
# Any first-degree relative with cancer at any site, from 2015 NHIS study.
FAMILY_HISTORY_AGE_BASE = {
    "21-29": 0.130,
    "30-39": 0.220,
    "40-49": 0.356,
    "50-59": 0.506,
    "60-69": 0.593,
    "70+": 0.550,
}
FAMILY_HISTORY_GROUP_MULTIPLIER = {
    "Hispanic": 21.2 / 35.6,
    "White": 42.0 / 35.6,
    "Black": 29.1 / 35.6,
    "Asian": 22.0 / 35.6,
    "Other": 1.0,
}

def age_band_for_hysterectomy(age_years: int) -> str:
    if age_years <= 29:
        return "21-29"
    if age_years <= 39:
        return "30-39"
    if age_years <= 49:
        return "40-49"
    if age_years <= 59:
        return "50-59"
    if age_years <= 69:
        return "60-69"
    return "70+"

def age_band_for_immuno(age_years: int) -> str:
    if age_years <= 29:
        return "21-29"
    if age_years <= 39:
        return "30-39"
    if age_years <= 49:
        return "40-49"
    if age_years <= 59:
        return "50-59"
    if age_years <= 69:
        return "60-69"
    if age_years <= 79:
        return "70-79"
    return "80+"

def age_band_for_prior_cancer(age_years: int) -> str:
    if age_years <= 44:
        return "21-44"
    if age_years <= 54:
        return "45-54"
    if age_years <= 64:
        return "55-64"
    if age_years <= 74:
        return "65-74"
    return "75+"

def age_band_for_family_history(age_years: int) -> str:
    if age_years <= 29:
        return "21-29"
    if age_years <= 39:
        return "30-39"
    if age_years <= 49:
        return "40-49"
    if age_years <= 59:
        return "50-59"
    if age_years <= 69:
        return "60-69"
    return "70+"


def age_band_for_insurance(age_years: int) -> str:
    if age_years <= 25:
        return "21-25"
    if age_years <= 34:
        return "26-34"
    if age_years <= 44:
        return "35-44"
    if age_years <= 54:
        return "45-54"
    if age_years <= 64:
        return "55-64"
    if age_years <= 74:
        return "65-74"
    return "75+"

def race_eth_group(race: str, ethnicity: str) -> str:
    # If the person is Hispanic, use the Hispanic-specific prevalence anchor.
    # Otherwise use the non-Hispanic race-specific anchor when available.
    # For smaller non-Hispanic race categories (AIAN, NHPI, Two or More), this script
    # currently falls back to an 'Other' bucket in the health-history modules because
    # the public prevalence tables we used later do not report those groups consistently.
    if ethnicity == "Hispanic":
        return "Hispanic"
    if race in {"White", "Black", "Asian"}:
        return race
    return "Other"

def clip_prob(x: float, lo: float = 0.0001, hi: float = 0.9999) -> float:
    return float(min(max(x, lo), hi))

def sample_age_years(rng: np.random.Generator, n: int) -> np.ndarray:
    base = rng.choice(AGE_VALUES, size=n, p=AGE_PROBS)
    age_years = base.astype(int).copy()

    # The age 85 bucket represents 85+ in the public source.
    mask_85plus = age_years == 85
    age_years[mask_85plus] = np.clip(
        np.round(rng.normal(loc=89.0, scale=4.0, size=mask_85plus.sum())).astype(int),
        85,
        100,
    )
    return age_years

def sample_race_ethnicity(rng: np.random.Generator, n: int):
    ethnicity = rng.choice(
        ["Hispanic", "Non-Hispanic"],
        size=n,
        p=[P_HISPANIC, P_NON_HISPANIC],
    )

    race = np.empty(n, dtype=object)
    hisp_idx = np.where(ethnicity == "Hispanic")[0]
    non_idx = np.where(ethnicity == "Non-Hispanic")[0]

    race[hisp_idx] = rng.choice(
        list(RACE_PROBS_HISP.keys()),
        size=len(hisp_idx),
        p=list(RACE_PROBS_HISP.values()),
    )
    race[non_idx] = rng.choice(
        list(RACE_PROBS_NON_HISP.keys()),
        size=len(non_idx),
        p=list(RACE_PROBS_NON_HISP.values()),
    )
    return race, ethnicity

def sample_blood_type(rng: np.random.Generator, n: int) -> np.ndarray:
    return rng.choice(
        list(BLOOD_TYPE_PROBS.keys()),
        size=n,
        p=list(BLOOD_TYPE_PROBS.values()),
    )


def sample_insurance_status(rng: np.random.Generator, age_years) -> np.ndarray:
    out = np.empty(len(age_years), dtype=object)
    for i, age in enumerate(age_years):
        band = age_band_for_insurance(int(age))
        probs = INSURANCE_PROBS[band]
        out[i] = rng.choice(
            ["Insured", "Uninsured"],
            p=[probs["Insured"], probs["Uninsured"]]
        )
    return out

def sample_bmi(rng: np.random.Generator, n: int) -> np.ndarray:
    # BMI source pages:
    #   - NYC obesity indicator page: https://a816-dohbesp.nyc.gov/IndicatorPublic/data-explorer/overweight/
    #   - NYC performance report with a 2024 adult-obesity benchmark: https://www.nyc.gov/assets/operations/downloads/pdf/mmr2025/dohmh.pdf
    # How BMI is generated in this script:
    #   1) draw whether the person is obese using the public obesity benchmark p = 0.276
    #   2) if not obese, draw BMI from a clipped Normal(24.7, 3.2) on [16.0, 29.9]
    #   3) if obese, draw BMI from a clipped Normal(34.8, 4.5) on [30.0, 60.0]
    # This is a synthetic mixture chosen to match the public obesity rate; it is not a direct
    # empirical BMI microdata sample from NYC.
    # First-pass BMI mixture calibrated so the obesity share is about 27.6%.
    obese = rng.binomial(1, 0.276, size=n).astype(bool)
    bmi = np.empty(n, dtype=float)

    vals0 = rng.normal(loc=24.7, scale=3.2, size=(~obese).sum())
    vals0 = np.clip(vals0, 16.0, 29.9)
    bmi[~obese] = vals0

    vals1 = rng.normal(loc=34.8, scale=4.5, size=obese.sum())
    vals1 = np.clip(vals1, 30.0, 60.0)
    bmi[obese] = vals1

    return np.round(bmi, 1)

# Generate Has Cervix from the hysterectomy lookup table.
# Formula used: P(Has Cervix = 1 | age, group) = 1 - P(Hysterectomy = 1 | age, group)
# For non-Hispanic AIAN, NHPI, and Two or More Races, the script currently uses an 'Other'
# hysterectomy profile defined as the midpoint-style average of the White, Black, and Asian lookup values.
def sample_has_cervix(rng: np.random.Generator, age_years, race, ethnicity) -> np.ndarray:
    out = np.empty(len(age_years), dtype=int)
    for i, age in enumerate(age_years):
        group = race_eth_group(race[i], ethnicity[i])
        age_band = age_band_for_hysterectomy(int(age))
        p_hyst = HYSTERECTOMY_BRFFS_2018[group][age_band]
        p_has_cervix = 1.0 - p_hyst
        out[i] = rng.binomial(1, clip_prob(p_has_cervix))
    return out

# Generate HIV/Immunocompromised from an age base rate times a demographic multiplier.
# Formula used: p = IMMUNO_AGE_BASE[age_band] * IMMUNO_GROUP_MULTIPLIER[group]
# For non-Hispanic AIAN, NHPI, and Two or More Races, the script currently uses the neutral
# 'Other' multiplier = 1.0 because the public source used here did not report those groups consistently.
def sample_immunocompromised(rng: np.random.Generator, age_years, race, ethnicity) -> np.ndarray:
    out = np.empty(len(age_years), dtype=int)
    for i, age in enumerate(age_years):
        group = race_eth_group(race[i], ethnicity[i])
        age_band = age_band_for_immuno(int(age))
        p = IMMUNO_AGE_BASE[age_band] * IMMUNO_GROUP_MULTIPLIER[group]
        out[i] = rng.binomial(1, clip_prob(min(p, 0.40)))
    return out

# Generate Prior Cancer Patient from an age base rate times a demographic multiplier.
# Formula used: p = PRIOR_CANCER_AGE_BASE[age_band] * PRIOR_CANCER_GROUP_MULTIPLIER[group]
# For non-Hispanic AIAN, NHPI, and Two or More Races, the script currently uses the neutral
# 'Other' multiplier = 1.0 because the public source used here did not report those groups consistently.
def sample_prior_cancer(rng: np.random.Generator, age_years, race, ethnicity) -> np.ndarray:
    out = np.empty(len(age_years), dtype=int)
    for i, age in enumerate(age_years):
        group = race_eth_group(race[i], ethnicity[i])
        age_band = age_band_for_prior_cancer(int(age))
        p = PRIOR_CANCER_AGE_BASE[age_band] * PRIOR_CANCER_GROUP_MULTIPLIER[group]
        out[i] = rng.binomial(1, clip_prob(min(p, 0.60)))
    return out

# Generate Family History from an age base rate times a demographic multiplier.
# Formula used: p = FAMILY_HISTORY_AGE_BASE[age_band] * FAMILY_HISTORY_GROUP_MULTIPLIER[group]
# For non-Hispanic AIAN, NHPI, and Two or More Races, the script currently uses the neutral
# 'Other' multiplier = 1.0 because the public source used here did not report those groups consistently.
def sample_family_history(rng: np.random.Generator, age_years, race, ethnicity) -> np.ndarray:
    out = np.empty(len(age_years), dtype=int)
    for i, age in enumerate(age_years):
        group = race_eth_group(race[i], ethnicity[i])
        age_band = age_band_for_family_history(int(age))
        p = FAMILY_HISTORY_AGE_BASE[age_band] * FAMILY_HISTORY_GROUP_MULTIPLIER[group]
        out[i] = rng.binomial(1, clip_prob(min(p, 0.95)))
    return out

def generate_patients(n: int = 28000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    patient_id = np.arange(1, n + 1)
    age_years = sample_age_years(rng, n)
    extra_days = rng.integers(0, 365, size=n)
    age_days = age_years * 365 + extra_days

    race, ethnicity = sample_race_ethnicity(rng, n)
    smoking = rng.binomial(1, P_SMOKING, size=n)
    alcohol = rng.binomial(1, P_ALCOHOL, size=n)
    blood_type = sample_blood_type(rng, n)
    bmi = sample_bmi(rng, n)

    has_cervix = sample_has_cervix(rng, age_years, race, ethnicity)
    hiv_immuno = sample_immunocompromised(rng, age_years, race, ethnicity)
    prior_cancer = sample_prior_cancer(rng, age_years, race, ethnicity)
    family_history = sample_family_history(rng, age_years, race, ethnicity)
    insurance_status = sample_insurance_status(rng, age_years)

    return pd.DataFrame({
        "Patient ID": patient_id,
        "Age (days)": age_days,
        "Race": race,
        "Ethnicity": ethnicity,
        "Smoking": smoking,
        "Alcohol": alcohol,
        "Blood Type": blood_type,
        "BMI": bmi,
        "Has Cervix": has_cervix,
        "HIV/Immunocompromised": hiv_immuno,
        "Prior Cancer Patient": prior_cancer,
        "Family History": family_history,
        "Insurance Status": insurance_status,
    })

def build_source_table() -> pd.DataFrame:
    expected_age_years_21_85 = float(np.sum(np.array(AGE_VALUES) * np.array(AGE_PROBS)))

    rows = [
        {
            "Attribute": "Patient ID",
            "Source": "Generated internally",
            "Metric used": "1, 2, 3, ..., n",
            "How calculated": "Sequential integers from 1 to n.",
            "Sampling / generation rule": "Deterministic sequence.",
            "Notes": "Not drawn from a public distribution."
        },
        {
            "Attribute": "Age (days)",
            "Source": "U.S. Census Bureau, SC-EST2024-AGESEX-CIV (NY female civilian population by single year of age)",
            "Metric used": f"Age-specific probabilities for ages 21-85; expected age over 21+ support = {expected_age_years_21_85:.2f} years before 85+ expansion",
            "How calculated": "Filter to NY females age 21+, then divide each age count by the total count across ages 21+ to get sampling probabilities.",
            "Sampling / generation rule": "Sample age in years from the empirical categorical distribution; convert to days as 365*years + Uniform(0,364). For the 85+ bucket, expand with a clipped Normal(89,4) to 85-100.",
            "Notes": "This keeps the synthetic age mix aligned with the public age structure."
        },
        {
            "Attribute": "Ethnicity",
            "Source": "U.S. Census Bureau, NY female sex/race/Hispanic-origin estimates",
            "Metric used": f"P(Hispanic)={P_HISPANIC:.4f}, P(Non-Hispanic)={P_NON_HISPANIC:.4f}",
            "How calculated": "Female Hispanic age-21+ count divided by the total female age-21+ count in the official Census 6-race-group state estimates; complement assigned to non-Hispanic.",
            "Sampling / generation rule": "Bernoulli/categorical draw.",
            "Notes": "Used as the first layer before race assignment."
        },
        {
            "Attribute": "Race",
            "Source": "U.S. Census Bureau, NY female sex/race/Hispanic-origin estimates",
            "Metric used": "Conditional race probabilities within Hispanic and Non-Hispanic groups",
            "How calculated": "Within each ethnicity group, compute shares of the official Census 6 race groups for New York females age 21+: White, Black, AIAN, Asian, NHPI, and Two or More Races.",
            "Sampling / generation rule": f'Hispanic: {RACE_PROBS_HISP}; Non-Hispanic: {RACE_PROBS_NON_HISP}',
            "Notes": "Hispanic/Latino is treated as ethnicity, not race. Race is stored separately using the Census 6-race-group classification."
        },
        {
            "Attribute": "Smoking",
            "Source": "NYC Health, adult smoking rate in NYC, 2023",
            "Metric used": f"P(Smoking)={P_SMOKING:.2f}",
            "How calculated": "Set equal to the published citywide adult smoking prevalence.",
            "Sampling / generation rule": "Bernoulli draw with p = published prevalence.",
            "Notes": "First-pass assumption applies the same rate across the 21+ female cohort."
        },
        {
            "Attribute": "Alcohol",
            "Source": "NYC Health, female adults reporting any alcohol in past 30 days, 2023",
            "Metric used": f"P(Alcohol)={P_ALCOHOL:.2f}",
            "How calculated": "Set equal to the published female prevalence of any drinking in the past 30 days.",
            "Sampling / generation rule": "Bernoulli draw with p = published prevalence.",
            "Notes": "This is any drinking, not heavy drinking."
        },
        {
            "Attribute": "Blood Type",
            "Source": "American Red Cross blood type frequency summaries",
            "Metric used": str({k: round(v, 4) for k, v in BLOOD_TYPE_PROBS.items()}),
            "How calculated": "Convert Red Cross approximate U.S. blood type frequencies into probabilities and normalize to sum to 1.",
            "Sampling / generation rule": "Categorical draw over the eight blood types.",
            "Notes": "Some source statements are approximate, such as '1 in 3' or 'less than 1%', so those were translated into close numeric values."
        },
        {
            "Attribute": "BMI",
            "Source": "NYC Health obesity prevalence benchmark",
            "Metric used": "Target obesity share P(BMI>=30)=0.276",
            "How calculated": "Not estimated from a full public BMI distribution. Instead, use a two-component mixture so that the simulated obesity rate matches the public benchmark.",
            "Sampling / generation rule": "First draw obese vs non-obese with p=0.276. If non-obese, BMI ~ clipped Normal(24.7, 3.2) on [16.0, 29.9]. If obese, BMI ~ clipped Normal(34.8, 4.5) on [30.0, 60.0].",
            "Notes": "The component means and SDs are modeling assumptions chosen to produce a plausible first-pass BMI distribution."
        },
        {
            "Attribute": "Has Cervix",
            "Source": "CDC/BRFSS-NHIS hysterectomy prevalence paper using 2018 BRFSS age-specific prevalence by race/ethnicity",
            "Metric used": "Hysterectomy prevalence by age band and race/ethnicity",
            "How calculated": "For each person, identify the age band and demographic group (Hispanic overrides race; otherwise use non-Hispanic White, Black, or Asian). Pull the 2018 BRFSS hysterectomy prevalence for that group.",
            "Sampling / generation rule": "P(Has Cervix=1) = 1 - P(Hysterectomy=1). Then draw Bernoulli.",
            "Notes": "For ages 70+, the oldest public age band in the source is used as the cap."
        },
        {
            "Attribute": "HIV/Immunocompromised",
            "Source": "2021 NHIS immunosuppression prevalence study",
            "Metric used": "Age-specific immunosuppression prevalence plus race/ethnicity multipliers",
            "How calculated": "Start from age-specific weighted immunosuppression prevalence, then multiply by the relative prevalence for Hispanic, White, Black, or Asian adults from the same study.",
            "Sampling / generation rule": "Bernoulli draw with p = age base × group multiplier.",
            "Notes": "This is a broad weakened-immune-system proxy and should be interpreted as an immunocompromised flag, not a pure HIV-only prevalence estimate."
        },
        {
            "Attribute": "Prior Cancer Patient",
            "Source": "CDC Health, United States history-of-cancer age percentages + NHIS 2018 crude cancer percentages by race/ethnicity",
            "Metric used": "Age-specific history-of-cancer percentages plus race/ethnicity multipliers",
            "How calculated": "Start from CDC age-group history-of-cancer percentages, then scale by the relative cancer prevalence for Hispanic, White, Black, or Asian adults using NHIS crude percentages.",
            "Sampling / generation rule": "Bernoulli draw with p = age base × group multiplier.",
            "Notes": "This is a first-pass self-reported prior-cancer flag, not site-specific cancer history."
        },
        {
            "Attribute": "Family History",
            "Source": "2015 NHIS family-history study",
            "Metric used": "Age-specific prevalence of reporting at least one first-degree relative with cancer at any site plus race/ethnicity multipliers",
            "How calculated": "Start from overall age-specific prevalence, then multiply by the relative prevalence for Hispanic, White, Black, or Asian adults reported in the study.",
            "Sampling / generation rule": "Bernoulli draw with p = age base × group multiplier.",
            "Notes": "This is a general family-history-of-cancer flag, not cancer-site-specific history."
        },
    ]
    return pd.DataFrame(rows)

def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = [
        {"metric": "n_patients", "value": len(df)},
        {"metric": "min_age_years", "value": round((df["Age (days)"] / 365.0).min(), 2)},
        {"metric": "mean_age_years", "value": round((df["Age (days)"] / 365.0).mean(), 2)},
        {"metric": "max_age_years", "value": round((df["Age (days)"] / 365.0).max(), 2)},
        {"metric": "mean_bmi", "value": round(df["BMI"].mean(), 2)},
        {"metric": "obesity_rate_bmi_ge_30", "value": round((df["BMI"] >= 30).mean(), 4)},
        {"metric": "smoking_rate", "value": round(df["Smoking"].mean(), 4)},
        {"metric": "alcohol_rate", "value": round(df["Alcohol"].mean(), 4)},
        {"metric": "has_cervix_rate", "value": round(df["Has Cervix"].mean(), 4)},
        {"metric": "immunocompromised_rate", "value": round(df["HIV/Immunocompromised"].mean(), 4)},
        {"metric": "prior_cancer_rate", "value": round(df["Prior Cancer Patient"].mean(), 4)},
        {"metric": "family_history_rate", "value": round(df["Family History"].mean(), 4)},
    ]

    for k, v in df["Race"].value_counts(normalize=True).sort_index().items():
        rows.append({"metric": f"race_{k}", "value": round(v, 4)})
    for k, v in df["Ethnicity"].value_counts(normalize=True).sort_index().items():
        rows.append({"metric": f"ethnicity_{k}", "value": round(v, 4)})
    for k, v in df["Blood Type"].value_counts(normalize=True).sort_index().items():
        rows.append({"metric": f"blood_{k}", "value": round(v, 4)})

    return pd.DataFrame(rows)

if __name__ == "__main__":
    df = generate_patients(n=28000, seed=42)
    source_df = build_source_table()
    summary_df = build_summary(df)


    with pd.ExcelWriter("synthetic_patients_21plus_28000_v4.xlsx", engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="patients", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        source_df.to_excel(writer, sheet_name="attribute_sources", index=False)

    print("Files written:")
    print("- synthetic_patients_21plus_28000_v4.csv")
    print("- synthetic_patients_21plus_28000_v4.xlsx")
    print("- synthetic_patients_21plus_summary_v4.csv")
    print("- synthetic_patients_attribute_sources_v4.csv")


Files written:
- synthetic_patients_21plus_28000_v4.csv
- synthetic_patients_21plus_28000_v4.xlsx
- synthetic_patients_21plus_summary_v4.csv
- synthetic_patients_attribute_sources_v4.csv
